# Module 3: Advanced Agents — State Control, HITL, & Tree Search (GCP Consolidated Edition)

### Objective
Welcome to Module 3. In this hands-on notebook, we will dive deep beneath the surface of autonomous agents using Google Cloud Platform (GCP). You will learn how to:
1. **Harness Human-in-the-Loop (HITL)** patterns using modern LangGraph `interrupt()` and `Command` workflows.
2. **Time Travel** through a graph's history to inspect, fork, and rewrite state checkpoints to fix failures dynamically.
3. **Orchestrate Language Agent Tree Search (LATS)** architectures using the unified Google GenAI SDK backend and Gemini 3.5 models.

In [ ]:
# Install the core LangGraph suite, consolidated Google GenAI integrations, and authentication tools
!pip install -U langgraph langchain-google-genai google-cloud-aiplatform

In [ ]:
import os
import sys

# 1. Authenticate to Google Cloud (skip if running inside a pre-authenticated Vertex AI Workbench)
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab user authenticated successfully.")

# 2. Configure your GCP Project variables
GCP_PROJECT = input("Enter your Google Cloud Project ID: ").strip()
GCP_LOCATION = input("Enter your GCP Region (e.g., us-central1): ").strip() or "us-central1"

# Point the modern ChatGoogleGenerativeAI class to your enterprise Vertex AI backend
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION

# Initialize the Vertex AI SDK for environmental tools
from google.cloud import aiplatform
aiplatform.init(project=GCP_PROJECT, location=GCP_LOCATION)
print(f"Vertex AI routing successfully initialized using the unified Google GenAI SDK.")

## Exercise 1: Controlled Generation & Dynamic Interrupts

### The Concept
In enterprise agent workflows, full autonomy can be a liability. We need deterministic guardrails. In modern LangGraph, the `interrupt()` function allows a node to immediately halt graph execution, save a checkpoint, and yield control back to the client thread.

To resume the graph, the client passes a `Command(resume=...)` object back into `graph.stream()`. This completely removes the legacy requirement of configuring hardcoded boundary conditions on compile time (`interrupt_before`/`interrupt_after`).

### Hands-on Task
Build a financial assistant that checks a user's transfer request. If the requested transaction amount is greater than $1,000, the agent must trigger a human approval workflow before execution.

In [ ]:
from typing import Annotated, Dict, Any, List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the modern consolidated Google LLM wrapper using Gemini 3.5 Flash
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", location="us")

# 1. Define the state shape
class FinancialState(TypedDict):
    messages: Annotated[list, add_messages]
    approved: bool
    amount: float
    recipient: str

# 2. Define the evaluation node
def check_transaction(state: FinancialState) -> Dict[str, Any]:
    print("--- CHECKING TRANSACTION ---")
    amount = state.get("amount", 1500.0)
    recipient = state.get("recipient", "Alice")

    if amount > 1000 and not state.get("approved", False):
        print(f"[Guardrail Triggered]: Request of ${amount} exceeds limit.")
        # interrupt() pauses execution immediately and serializes this message out to the client
        human_response = interrupt(
            {
                "reason": "Transaction exceeds manual approval threshold.",
                "amount": amount,
                "recipient": recipient
            }
        )

        # Process the resume command value
        if human_response == "Approved":
            return {"approved": True, "messages": [AIMessage(content="Human manager approved the transfer.")]}
        else:
            return {"approved": False, "messages": [AIMessage(content="Human manager rejected the transfer.")]}

    return {"approved": True}

# 3. Define the action node
def execute_transfer(state: FinancialState) -> Dict[str, Any]:
    print("--- EXECUTING TRANSACTION ---")
    if state.get("approved"):
        return {"messages": [AIMessage(content=f"Successfully transferred ${state['amount']} to {state['recipient']}.")]}
    else:
        return {"messages": [AIMessage(content="Transaction aborted due to lack of clearance.")]}

# 4. Build the graph
workflow = StateGraph(FinancialState)
workflow.add_node("check_transaction", check_transaction)
workflow.add_node("execute_transfer", execute_transfer)

workflow.add_edge(START, "check_transaction")
workflow.add_edge("check_transaction", "execute_transfer")
workflow.add_edge("execute_transfer", END)

# Compile with persistence memory to support interrupts
memory = MemorySaver()
financial_graph = workflow.compile(checkpointer=memory)

In [ ]:
# --- Production-Ready Live Interactive Driver Code Cell ---

# 1. Initialize our thread tracking config and base transactional metadata
config = {"configurable": {"thread_id": "gcp_live_interactive_session"}}
current_input = {
    "messages": [HumanMessage(content="Wire $1500 to Bob for consulting work.")],
    "amount": 1500.0,
    "recipient": "Bob",
    "approved": False
}

print("--- Starting Live Interactive Pipeline Execution ---")

# 2. Open an ongoing processing loop that yields live input prompts when requested
while True:
    has_interrupted = False

    # Run the streaming engine using our current active inputs
    for chunk in financial_graph.stream(current_input, config, stream_mode="values"):
        if chunk.get("messages"):
            print(f"Agent: {chunk['messages'][-1].content}")

    # Post-stream checkpoint check: Is the engine currently suspended?
    snapshot = financial_graph.get_state(config)

    if snapshot.tasks and snapshot.tasks[0].interrupts:
        # Extract the exact payload passed to the interrupt() method in your graph node
        interrupt_data = snapshot.tasks[0].interrupts[0].value

        print(f"\n⚠️  [PAUSED BY REASON]: {interrupt_data.get('reason')}")
        print(f"   Request: Transferred ${interrupt_data.get('amount')} -> {interrupt_data.get('recipient')}")

        # Open an interactive terminal text entry widget inside the Jupyter notebook cell
        user_feedback = input("\nEnter human authorization verdict ('Approved' or 'Rejected'): ")

        # Format a resumption command payload to kick off the next loop pass
        current_input = Command(resume=user_feedback)
        has_interrupted = True
        print("\n--- Resuming graph execution timeline with human feedback... ---\n")

    # If the loop completes a stream pass without encountering a task interrupt, we've hit the END node
    if not has_interrupted:
        print("\n✅ Execution Pipeline Terminated Successfully.")
        break

## Exercise 2: Time Travel (Replaying & Forking Past States)

### The Concept
Because LangGraph saves explicit state snapshots (checkpoints) at every node transition, we can browse through a thread's history using `graph.get_state_history()`.

If an agent runs off the rails because a tool returned an unexpected BigQuery syntax error or bad cloud data payload, we can programmatically:
1. Look back in time to the checkpoint right *before* things went wrong.
2. Edit or override that historic state with a corrected payload using `graph.update_state()`.
3. Fork a new alternate path forward using that corrected checkpoint, saving us from having to rerun the entire pipeline from scratch.

### Hands-on Task
Simulate a scenario where a database tool crashes due to a bad syntax string payload. You will fetch history, update the state to correct the query, and resume execution cleanly down a successful fork.

In [ ]:
class BuggyState(TypedDict):
    messages: Annotated[list, add_messages]
    query: str

def code_generator(state: BuggyState) -> Dict[str, Any]:
    print("--- NODE: Code Generator ---")
    return {"query": "SELECT * FROM `gcp_project.dataset.users` WHERE age > 'eighteen'"} # Intentional semantic bug

def database_runner(state: BuggyState) -> Dict[str, Any]:
    print("--- NODE: Database Runner ---")
    query = state["query"]
    # Check for our intentional data-type error
    if "eighteen" in query:
        raise ValueError(f"BigQuery Error: Invalid type for age field (expected INT64, got STRING) in query: '{query}'")
    return {"messages": [AIMessage(content=f"Executed query successfully on BigQuery. Found 42 rows.")]}

# Build the pipeline
buggy_workflow = StateGraph(BuggyState)
buggy_workflow.add_node("generator", code_generator)
buggy_workflow.add_node("runner", database_runner)
buggy_workflow.add_edge(START, "generator")
buggy_workflow.add_edge("generator", "runner")
buggy_workflow.add_edge("runner", END)

buggy_graph = buggy_workflow.compile(checkpointer=MemorySaver())

In [ ]:
config_tt = {"configurable": {"thread_id": "gcp_unified_time_travel"}}

# 1. Execute the graph and witness the failure crash
print("=========================================================")
print("STEP 1: Kicking off initial graph run with broken payload")
print("=========================================================")
try:
    buggy_graph.invoke({"messages": [HumanMessage(content="Fetch all adults.")], "query": ""}, config_tt)
except Exception as e:
    print(f"\n💥 EXECUTION CRASHED AS EXPECTED!")
    print(f"Error Message Caught: {e}\n")

# 2. Inspect the current active thread state
print("=========================================================")
print("STEP 2: Targeted Checkpoint Inspection via get_state()")
print("=========================================================")
print("💡 UNDER THE HOOD:")
print("Because the 'database_runner' node threw an unhandled exception,")
print("LangGraph immediately aborted the transaction. Crucially, it did NOT")
print("commit a new checkpoint for the failure. This means the current")
print("active checkpoint on this thread is automatically frozen at the exact")
print("moment right BEFORE the crashing node attempted to run.")

current_state = buggy_graph.get_state(config_tt)

print(f"\n🎯 [Targeted Checkpoint ID]: {current_state.config['configurable']['checkpoint_id']}")
print(f"📋 [Next Node in Queue]:      {current_state.next} <-- Frozen right before entry!")
print(f"❌ [Current Broken State]:    query = \"{current_state.values.get('query')}\"")
print("-" * 57)

# 3. Fork the timeline by injecting the corrected state
print("\n=========================================================")
print("STEP 3: Updating State & Forking the Checkpoint Timeline")
print("=========================================================")
print("💡 UNDER THE HOOD:")
print("We pass our targeted checkpoint config into update_state(). LangGraph")
print("takes the state at that exact point in time, applies our query fix,")
print("and writes a BRAND NEW checkpoint. It returns a new configuration")
print("dictionary ('forked_config') containing our new alternate branch ID.")

forked_config = buggy_graph.update_state(
    current_state.config,
    {"query": "SELECT * FROM `gcp_project.dataset.users` WHERE age > 20"}, # Injecting our integer fix
    as_node="generator"
)

print(f"\n🌱 [Original Checkpoint ID]: {current_state.config['configurable']['checkpoint_id']}")
print(f"🚀 [New Forked Checkpoint ID]: {forked_config['configurable']['checkpoint_id']}")
print("The state has been successfully corrected on our new timeline branch!")
print("-" * 57)

# 4. Resume execution down our newly minted fork
print("\n=========================================================")
print("STEP 4: Resuming Execution Down the Fixed Fork")
print("=========================================================")
print("💡 UNDER THE HOOD:")
print("We call invoke() passing 'None' as the input. This tells LangGraph")
print("to look at the state already waiting in the checkpointer. By passing")
print("our 'forked_config', it bypasses the crashed branch completely and")
print("resumes the runner node using our corrected integer query.")

print("\n--- Resuming from the forked turning point ---")
final_state = buggy_graph.invoke(None, forked_config)

print(f"\n✅ SUCCESS!")
print(f"Final Agent Output: {final_state['messages'][-1].content}")
print("=========================================================")